In [ ]:
!pip install -q transformers pandas scikit-learn
import os
import json
import re
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score, cohen_kappa_score
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
DATA_PATH = "/content/drive/MyDrive/CS4248/"
ANNOTATIONS_FILE = os.path.join(DATA_PATH, "lstm/training_data.csv")
TEXT_COL = "clean_text" 
LABEL_COL = "sentiment"
LABEL_NAMES = ["negative", "neutral", "positive"]

# Hyperparameters
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_LEN = 64

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps') # M1/M2/M3 Mac GPU support
else:
    DEVICE = torch.device('cpu')

In [ ]:
TOKENIZER_TYPE = 'word'

class CustomWordTokenizer:
    """A classic word-level tokenizer that builds a vocabulary based on word frequency."""
    def __init__(self, max_vocab_size=20000):
        self.max_vocab_size = max_vocab_size
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.vocab_size = 2
        
    def fit(self, texts):
        counter = Counter()
        for text in texts:
            words = str(text).lower().split()
            counter.update(words)
            
        for word, _ in counter.most_common(self.max_vocab_size - 2):
            self.word2idx[word] = self.vocab_size
            self.idx2word[self.vocab_size] = word
            self.vocab_size += 1
            
    def encode_plus(self, text, max_length, **kwargs):
        words = str(text).lower().split()
        input_ids = [self.word2idx.get(w, self.word2idx['<UNK>']) for w in words]
        
        if len(input_ids) > max_length:
            input_ids = input_ids[:max_length]
        else:
            input_ids = input_ids + [self.word2idx['<PAD>']] * (max_length - len(input_ids))
            
        return {
            'input_ids': torch.tensor(input_ids).unsqueeze(0)
        }

# --- DATASET COMPONENT ---
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.label_map = {name: i for i, name in enumerate(LABEL_NAMES)}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = int(self.labels[item] if pd.notnull(self.labels[item]) else 1)

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'targets': torch.tensor(label, dtype=torch.long)
        }

# --- MODEL DEFINITION (BiLSTM + Multi-Head Attention, weighted-sum pooling) ---
class LSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_classes=3,
                 num_layers=2, bidirectional=True, dropout=0.5, num_heads=8,
                 pretrained_embeddings=None, freeze_embeddings=False):
        super(LSTMAttentionClassifier, self).__init__()
        if pretrained_embeddings is not None:
            embedding_dim = pretrained_embeddings.shape[1]
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_embeddings, freeze=freeze_embeddings, padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                            bidirectional=bidirectional, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)

        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim

        # Multi-head self-attention
        self.attention = nn.MultiheadAttention(
            embed_dim=lstm_output_dim, num_heads=num_heads,
            batch_first=True, dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_output_dim, num_classes)

    def forward(self, input_ids):
        pad_mask = (input_ids == 0)                                       # (batch, seq_len) True = padding

        embedded = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embedded)                                 # (batch, seq_len, hidden*2)

        # Multi-head attention — return averaged weights across heads
        attn_out, attn_weights = self.attention(
            lstm_out, lstm_out, lstm_out,
            key_padding_mask=pad_mask,
            need_weights=True,
            average_attn_weights=True,
        )                                                                  # attn_weights: (batch, seq_len, seq_len)

        # Weighted sum: use per-query averaged attention weight as importance score
        # Sum over key dimension to get per-position importance, then normalise
        token_scores = attn_weights.sum(dim=1)                            # (batch, seq_len)
        token_scores = token_scores.masked_fill(pad_mask, 0.0)
        token_scores = token_scores / token_scores.sum(dim=1, keepdim=True).clamp(min=1e-9)

        context = (token_scores.unsqueeze(-1) * lstm_out).sum(dim=1)     # (batch, hidden*2)

        out = self.dropout(context)
        out = self.fc(out)
        return out

# --- TRAINING LOOP ---
def train_epoch(model, data_loader, loss_fn, optimizer, device, clip_grad=1.0):
    model = model.train()
    losses = []
    correct_predictions = 0

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        targets = batch['targets'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids)
        _, preds = torch.max(outputs, dim=1)
        loss = loss_fn(outputs, targets)

        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())

        loss.backward()
        if clip_grad:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

def eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses = []
    correct_predictions = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            targets = batch['targets'].to(device)

            outputs = model(input_ids)
            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, targets)

            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    avg_acc = correct_predictions.double() / len(data_loader.dataset)
    return avg_acc, np.mean(losses), all_targets, all_preds

# --- GLOVE LOADER ---
def load_glove_embeddings(glove_path, word2idx, embedding_dim):
    """Load pretrained GloVe vectors. Missing words get small random vectors instead of zeros."""
    embeddings = np.random.uniform(-0.05, 0.05, (len(word2idx), embedding_dim))
    embeddings[0] = 0  # <PAD> stays zero

    found = 0
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            if word in word2idx:
                embeddings[word2idx[word]] = np.array(values[1:], dtype='float32')
                found += 1
    print(f"GloVe: {found}/{len(word2idx)} vocab words found ({found/len(word2idx)*100:.1f}%)")
    return torch.FloatTensor(embeddings)

In [ ]:
print(f"Using device: {DEVICE}")
if not os.path.exists(ANNOTATIONS_FILE):
    print(f"Error: {ANNOTATIONS_FILE} not found. Please check your DATA_PATH/Google Drive mount.")

print("Loading dataset...")
df = pd.read_csv(ANNOTATIONS_FILE)
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df[LABEL_COL])
print(f"Train size: {len(df_train)}, Test size: {len(df_test)}")

if TOKENIZER_TYPE == 'subword':
    print("\n--- Initializing Hugging Face AutoTokenizer (Subword) ---")
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    vocab_size = tokenizer.vocab_size
else:
    print("\n--- Initializing Custom Classic Vocab Tokenizer (Word-level) ---")
    tokenizer = CustomWordTokenizer(max_vocab_size=20000)
    tokenizer.fit(df_train[TEXT_COL].values)
    vocab_size = tokenizer.vocab_size

train_dataset = TweetDataset(df_train[TEXT_COL].values, df_train[LABEL_COL].values, tokenizer, MAX_LEN)
test_dataset  = TweetDataset(df_test[TEXT_COL].values,  df_test[LABEL_COL].values,  tokenizer, MAX_LEN)

train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_data_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Vocab size: {vocab_size}")

In [ ]:
model = LSTMAttentionClassifier(vocab_size=vocab_size, embedding_dim=128, hidden_dim=256, num_classes=3)
model = model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss().to(DEVICE)

best_acc = 0

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("-" * 10)

    train_acc, train_loss = train_epoch(model, train_data_loader, loss_fn, optimizer, DEVICE)
    print(f"Train loss {train_loss:.4f} accuracy {train_acc:.4f}")

    val_acc, val_loss, targets, preds = eval_model(model, test_data_loader, loss_fn, DEVICE)
    print(f"Val   loss {val_loss:.4f} accuracy {val_acc:.4f}")
    print()

    if val_acc > best_acc:
        best_acc = val_acc

# Final evaluation
print(f"Final Evaluation on Test Set ({TOKENIZER_TYPE} tokenization):")
_, _, final_targets, final_preds = eval_model(model, test_data_loader, loss_fn, DEVICE)
print(classification_report(final_targets, final_preds, target_names=LABEL_NAMES))

In [ ]:
# --- TRAIN ON FULL TRAINING SET, EVALUATE ON tweets.csv ---
# Uses GloVe Twitter embeddings (200d) + BiLSTM + Multi-Head Attention (8 heads)
TWEETS_FILE = os.path.join(DATA_PATH, "lstm/aggregated.csv")
TWEETS_TEXT_COL = "clean_txt"
TWEETS_LABEL_COL = "label"
GLOVE_DIM = 200
EPOCHS = 20
DROPOUT = 0.5
FREEZE_GLOVE_EMBEDDINGS = True
NUM_HEADS = 8
SEED = 42

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_LEN = 64

AUGMENTED_FILE = os.path.join(DATA_PATH, "training_data_augmented.csv")
PREDS_OUTPUT = os.path.join(DATA_PATH, "lstm/bilstm_attention_preds.csv")

# --- Reproducibility ---
import random, copy, zipfile
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GLOVE_DIR = os.path.join(DATA_PATH, "glove")
GLOVE_FILE = os.path.join(GLOVE_DIR, f"glove.twitter.27B.{GLOVE_DIM}d.txt")
os.makedirs(GLOVE_DIR, exist_ok=True)
if not os.path.exists(GLOVE_FILE):
    print("Downloading GloVe Twitter embeddings to Drive (~1.4 GB)...")
    GLOVE_ZIP = os.path.join(GLOVE_DIR, "glove.twitter.27B.zip")
    os.system(f"wget -q https://nlp.stanford.edu/data/glove.twitter.27B.zip -O {GLOVE_ZIP}")
    with zipfile.ZipFile(GLOVE_ZIP, 'r') as z:
        z.extract(f"glove.twitter.27B.{GLOVE_DIM}d.txt", GLOVE_DIR)
    os.remove(GLOVE_ZIP)
    print("Done. Saved to Drive.")
else:
    print(f"GloVe found in Drive: {GLOVE_FILE}")

print("\nLoading datasets...")
df_full = pd.read_csv(AUGMENTED_FILE)
df_full = df_full.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df_tweets = pd.read_csv(TWEETS_FILE)
df_tweets = df_tweets.dropna(subset=[TWEETS_TEXT_COL, TWEETS_LABEL_COL]).copy()
print(f"Full training size: {len(df_full)}  |  Tweets test size: {len(df_tweets)}")
print("Class distribution (train):")
print(df_full[LABEL_COL].value_counts().sort_index())

full_tokenizer = CustomWordTokenizer(max_vocab_size=20000)
full_tokenizer.fit(df_full[TEXT_COL].values)
print(f"Vocab size: {full_tokenizer.vocab_size}")
glove_embeddings = load_glove_embeddings(GLOVE_FILE, full_tokenizer.word2idx, GLOVE_DIM)

# Datasets and loaders
full_train_dataset  = TweetDataset(df_full[TEXT_COL].values, df_full[LABEL_COL].values, full_tokenizer, MAX_LEN)
tweets_test_dataset = TweetDataset(df_tweets[TWEETS_TEXT_COL].values, df_tweets[TWEETS_LABEL_COL].values, full_tokenizer, MAX_LEN)
full_train_loader   = DataLoader(full_train_dataset,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
tweets_test_loader  = DataLoader(tweets_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 1-layer BiLSTM + Multi-Head Attention (8 heads), hidden=128, frozen GloVe
full_model = LSTMAttentionClassifier(
    vocab_size=full_tokenizer.vocab_size,
    hidden_dim=128,
    num_classes=3,
    num_layers=1,
    bidirectional=True,
    dropout=DROPOUT,
    num_heads=NUM_HEADS,
    pretrained_embeddings=glove_embeddings,
    freeze_embeddings=FREEZE_GLOVE_EMBEDDINGS,
)
full_model = full_model.to(DEVICE)

full_optimizer = torch.optim.Adam(full_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Weighted loss: penalise neg/pos errors more to match test distribution (162 neutral vs 114/126)
class_weights = torch.tensor([1.4, 1.0, 1.3], dtype=torch.float).to(DEVICE)
full_loss_fn = nn.CrossEntropyLoss(weight=class_weights)
print(f"Class weights: negative={class_weights[0]:.1f}, neutral={class_weights[1]:.1f}, positive={class_weights[2]:.1f}")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(full_optimizer, mode='min', patience=2, factor=0.5)

print(f"\nTraining on augmented dataset for {EPOCHS} epochs (GloVe-{GLOVE_DIM}d, frozen, BiLSTM + {NUM_HEADS}-head attention, weighted loss, seed={SEED})...")
epoch_log = []
best_f1_so_far = 0.0
best_model_state = None

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("-" * 10)
    train_acc, train_loss = train_epoch(full_model, full_train_loader, full_loss_fn, full_optimizer, DEVICE, clip_grad=1.0)
    print(f"Train loss {train_loss:.4f} accuracy {train_acc:.4f}")
    scheduler.step(train_loss)

    _, _, t_targets, t_preds = eval_model(full_model, tweets_test_loader, full_loss_fn, DEVICE)
    f1 = f1_score(t_targets, t_preds, average='macro')
    acc = accuracy_score(t_targets, t_preds)
    epoch_log.append((epoch + 1, f1, acc))
    print(f"  >> Epoch {epoch + 1} | Test Macro F1: {f1:.4f} | Accuracy: {acc:.4f}")

    if f1 > best_f1_so_far:
        best_f1_so_far = f1
        best_model_state = copy.deepcopy(full_model.state_dict())

print("\n--- Epoch checkpoint summary ---")
print(f"{'Epoch':>6}  {'Macro F1':>9}  {'Accuracy':>9}")
for ep, f1, acc in epoch_log:
    print(f"{ep:>6}  {f1:>9.4f}  {acc:>9.4f}")

best_ep, best_f1, best_acc = max(epoch_log, key=lambda x: x[1])
print(f"\nBest: Epoch {best_ep} — Macro F1 {best_f1:.4f}, Accuracy {best_acc:.4f}")

# Restore best checkpoint before final eval and CSV export
full_model.load_state_dict(best_model_state)
print(f"\nRestored best model (epoch {best_ep})")

print(f"\nFinal Evaluation on tweets.csv (best epoch: {best_ep}):")
_, _, tweets_targets, tweets_preds = eval_model(full_model, tweets_test_loader, full_loss_fn, DEVICE)
print(classification_report(tweets_targets, tweets_preds, target_names=LABEL_NAMES))
print(f"Accuracy:      {accuracy_score(tweets_targets, tweets_preds):.4f}")
print(f"Macro F1:      {f1_score(tweets_targets, tweets_preds, average='macro'):.4f}")
print(f"Cohen Kappa:   {cohen_kappa_score(tweets_targets, tweets_preds):.4f}")

# --- Save predictions in standard format ---
full_model.eval()
all_probs = []
with torch.no_grad():
    for batch in tweets_test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        logits = full_model(input_ids)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)

all_probs = np.vstack(all_probs)
preds_df = pd.DataFrame({
    'pred':          all_probs.argmax(axis=1),
    'prob_negative': all_probs[:, 0],
    'prob_neutral':  all_probs[:, 1],
    'prob_positive': all_probs[:, 2],
    'confidence':    all_probs.max(axis=1),
})
preds_df.to_csv(PREDS_OUTPUT, index=False)
print(f"\nPredictions saved to {PREDS_OUTPUT}")